# Día 9 — Optimización de *Running Total* con `SUM() OVER()`

En este entregable se compara una consulta **antes** y **después** para calcular un **total acumulado** de `Tutoring_Sessions` por nivel de involucramiento parental.

## ¿Qué hace `SUM() OVER()`?

`SUM() OVER(...)` es una **función de ventana** que permite calcular agregados sin perder el detalle de cada fila.

En este caso:

- `PARTITION BY Parental_Involvement` separa los datos por grupo (`Low`, `Medium`, `High`).
- `ORDER BY student_id` define el orden dentro de cada grupo.
- `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` acumula desde la primera fila del grupo hasta la fila actual.

Resultado: para cada estudiante se obtiene su valor individual y, además, el **total acumulado** del grupo al que pertenece.

## Resumen de la optimización (Antes vs Después)

- **Antes**: uso de `SELECT *` y sin filtro temprano → más filas/columnas procesadas, mayor consumo de memoria y tiempo.
- **Después**:
    1. Selección explícita de columnas necesarias.
    2. Filtro en `WHERE` antes de la ventana.
    3. Menor I/O y mejor rendimiento para escenarios de producción.

En síntesis, se mantiene la misma lógica analítica del acumulado, pero con una ejecución más eficiente.


In [16]:
import duckdb
import os
os.listdir('../data/student_performance')
con = duckdb.connect()
con.execute("CREATE TABLE student_performance AS SELECT * FROM read_csv_auto('../data/student_performance/student_performance.csv')")
result = con.execute("SELECT * FROM student_performance").fetchdf()
print(result.head())


   student_id  Hours_Studied  Attendance Parental_Involvement  \
0           1             23          84                  Low   
1           2             19          64                  Low   
2           3             24          98               Medium   
3           4             29          89                  Low   
4           5             19          92               Medium   

  Access_to_Resources  Extracurricular_Activities  Sleep_Hours  \
0                High                       False            7   
1              Medium                       False            8   
2              Medium                        True            7   
3              Medium                        True            8   
4              Medium                        True            6   

   Previous_Scores Motivation_Level  Internet_Access  ...  Family_Income  \
0               73              Low             True  ...            Low   
1               59              Low             True  ...   

## Version ineficiente

In [18]:
con.execute("""
SELECT *,
    SUM(tutoring_sessions) OVER (
        PARTITION BY parental_involvement
        ORDER BY student_id
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS tutoring_acumulado
FROM student_performance;
""").fetchdf()

,student_id,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,...,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score,tutoring_acumulado
0,24,11,78,High,Medium,True,8,100,High,True,...,Medium,Public,Neutral,3,False,High School,Moderate,Male,66,1.0
1,26,21,62,High,Low,True,6,54,High,True,...,High,Public,Positive,3,False,Postgraduate,Far,Male,64,1.0
2,28,22,83,High,High,True,6,94,Medium,True,...,Medium,Public,Neutral,2,False,College,Moderate,Male,71,1.0
3,30,18,66,High,High,False,4,51,Low,True,...,Medium,Private,Neutral,2,False,College,Near,Male,64,3.0
4,34,14,60,High,Medium,False,5,50,Medium,True,...,Medium,Public,Neutral,3,False,College,NaN,Female,61,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6602,4030,25,75,Medium,Low,True,8,75,High,True,...,High,Private,Negative,4,True,Postgraduate,Near,Male,68,3014.0
6603,4031,25,99,Medium,Medium,False,8,83,Medium,True,...,Medium,Public,Neutral,2,False,College,Near,Female,72,3015.0
6604,4035,15,87,Medium,Medium,False,10,83,Medium,True,...,High,Private,Positive,3,False,High School,Near,Male,68,3016.0
6605,4037,23,78,Medium,Medium,True,5,83,Medium,True,...,Medium,Public,Positive,3,False,High School,Near,Male,68,3017.0


Version eficiente

In [21]:
con.execute("""
SELECT
    student_id,
    parental_involvement,
    tutoring_sessions,
    SUM(tutoring_sessions) OVER (
        PARTITION BY parental_involvement
        ORDER BY student_id
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS tutoring_acumulado
FROM student_performance
WHERE parental_involvement IN ('Low', 'Medium', 'High')  -- filtra explícitamente
ORDER BY parental_involvement, student_id;
""").fetchdf()

,student_id,Parental_Involvement,Tutoring_Sessions,tutoring_acumulado
0,24,High,1,1.0
1,26,High,0,1.0
2,28,High,0,1.0
3,30,High,2,3.0
4,34,High,2,5.0
...,...,...,...,...
6602,6600,Medium,3,4975.0
6603,6601,Medium,2,4977.0
6604,6602,Medium,2,4979.0
6605,6605,Medium,3,4982.0
